# 03 — Model Evaluation

Reproduces Table 2 (Peru-wide and validation-region overlap/AUC for all five models) and the per-glacier analysis figures: overlap histogram and vulnerability characteristics.

All logic lives in `glacier_melt.evaluate` and `glacier_melt.visualise`; this notebook only loads predictions, calls those functions, and displays results.

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from glacier_melt.sampling import LABEL_COL
from glacier_melt.evaluate import (
    evaluate_model, print_summary_table,
    assign_pixels_to_glaciers, per_glacier_metrics,
    glacier_physical_features, vulnerability_ranking,
)
from glacier_melt.visualise import (
    plot_overlap_histogram, plot_vulnerability_characteristics,
)

PREDICTIONS_DIR = Path("../data/Predictions")
RGI_PATH = Path("../data/rgi70_peru.geojson")
OUTPUT_DIR = Path("../results")
OUTPUT_DIR.mkdir(exist_ok=True)

VALIDATION_REGION = "r3"

## Load predictions and RGI polygons

In [ ]:
df = pd.read_parquet(
    PREDICTIONS_DIR / "predictions_full_peru_r3holdout.parquet"
)
rgi_gdf = gpd.read_file(RGI_PATH)

region_mask = (df["region"] == VALIDATION_REGION).values
print(f"Loaded {len(df):,} pixels, {region_mask.sum():,} in validation region")

## Table 2 — Peru-wide and validation metrics for all models

In [ ]:
model_cols = {
    "RF (EASD)":   "rf_easd_prob",
    "RF (AE)":     "rf_ae_prob",
    "MLP (AE)":    "mlp_prob",
    "CNN 3x3 (AE)": "cnn3x3_prob",
    "CNN 5x5 (AE)": "cnn5x5_prob",
}

results = {}
for name, col in model_cols.items():
    results[name] = evaluate_model(
        df[col].values, df[LABEL_COL].values,
        region_mask=region_mask, model_name=name,
    )

print_summary_table(results)

## Per-glacier analysis (selected model: MLP)

Assigns pixels to RGI glaciers, computes per-glacier overlap, and joins physical features for the vulnerability characteristics plot.

In [ ]:
df["rgi_id"] = assign_pixels_to_glaciers(df, rgi_gdf)

df_overlap = per_glacier_metrics(
    df, df["mlp_prob"].values, min_area_km2=0.1,
)
df_overlap.to_parquet(OUTPUT_DIR / "per_glacier_overlap.parquet", index=False)

### Overlap histogram — train vs validation regions

Glaciers smaller than 0.1 km² are filtered out, since complete melting of
small glaciers produces a melt_rate of 1.0 and trivially inflates overlap
regardless of model spatial accuracy.

In [ ]:
fig = plot_overlap_histogram(
    df_overlap,
    df[["rgi_id", "region"]].drop_duplicates(subset="rgi_id"),
    val_region=VALIDATION_REGION,
)
fig.savefig(OUTPUT_DIR / "overlap_histogram.png", dpi=200, bbox_inches="tight")
plt.show()

## Vulnerability ranking and physical characteristics

Ranks all glaciers by mean MLP predicted melt probability, then compares
physical characteristics (elevation, slope, edge distance, area) between
the most and least vulnerable quartiles. Demonstrates that vulnerability
is primarily driven by glacier size and edge geometry, not slope.

In [ ]:
df_vuln = vulnerability_ranking(df, df["mlp_prob"].values)
df_physical = glacier_physical_features(df)
df_vuln = df_vuln.merge(
    df_physical.drop(columns=["n_pixels", "area_km2"]), on="rgi_id", how="left"
)
df_vuln.to_parquet(OUTPUT_DIR / "vulnerability_ranking.parquet", index=False)

df_vuln.head(10)

In [ ]:
fig = plot_vulnerability_characteristics(df_vuln)
fig.savefig(OUTPUT_DIR / "vulnerability_characteristics.png", dpi=200, bbox_inches="tight")
plt.show()